In [1]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langgraph.graph import MessagesState, StateGraph
from typing import List, Union
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_chroma import Chroma
import os
from langgraph.types import Command
from langchain.messages import HumanMessage

d:\programming\LangChain impelentation\LangGraph\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
current_dir=os.getcwd()
persistant_dir=os.path.join(current_dir,"database","Chroma_db")

In [3]:
embedding=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

db=Chroma(
    persist_directory=persistant_dir,
    embedding_function=embedding,
    collection_metadata={"hnsw_space":"cosine"}
)

retriver=db.as_retriever(search_type='mmr',
                search_kwargs={
                    "k":3,
                    "fetch_k":10,
                    "lambda_multi":0.5
                })


In [4]:

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

bouncer_llm_prompt=ChatPromptTemplate([
    ("system","You are a decision maker AI assistant. Your duty is to identify if the users question is "
    "about one of the following topics. \n\n"
    "1. RAG\n"
    "2. The foundational architecture of Retrieval-Augmented Generation (RAG) systems\n'"
    "3. The core components of RAG pipelines\n"
    "4. The classification of user queries in RAG systems\n"
    "5. Training paradigms and knowledge injection strategies for RAG models\n"
    "6. Methodologies and datasets utilized for systematically evaluating RAG systems\n"
    "7. Advanced methodologies for enhancing traditional RAG frameworks\n"
    "8. Future opportunities, ethical challenges, and the development of trustworthy RAG\n"
    ""
    "If the users question is about one of this topic respond with 'Yes' otherwise repond 'No'."),
    MessagesPlaceholder(variable_name="messages")
])

responder_llm_prompt=ChatPromptTemplate([
    ("system","You are an AI Assistnt. Your duty is to provide answer to the user query based on the "
    "provided documents.\n\n"
    "DOCUMENTS:\n"
    "{documents}"),
    MessagesPlaceholder(variable_name="messages")
])

In [5]:
class AgentState(MessagesState):
    documents:Union[List[Document],None]
    on_topic:str

In [6]:
answer_generator_chain=responder_llm_prompt | llm
bouncer_chain=bouncer_llm_prompt | llm

def bouncer_node(state:AgentState):
    decision=bouncer_chain.invoke(state)
    decision=decision.content.lower()
    if "yes" in decision:
        return Command(
            goto="retriver",
            update={"on_topic":"Yes"}
        )

    else:
        return Command(
            goto="off_topic_responder",
            update={"on_topic":"No"}
        )

def retriver_node(state:AgentState):
    query=state["messages"][-1].content
    retrived_data=retriver.invoke(query)
    return {"documents":retrived_data}

def generate_answer_node(state:AgentState):
    answer=answer_generator_chain.invoke(state)
    return {"messages":answer}

In [7]:
graph=StateGraph(AgentState)
graph.add_node("answer_generator",generate_answer_node)
graph.add_node("retriver",retriver_node)
graph.add_node("bouncer",bouncer_node)

graph.add_edge("retriver","answer_generator")
graph.set_entry_point("bouncer")

app=graph.compile()

In [10]:
query="What are the fundamental differences between utilizing a foundation model, LLM fine-tuning, and a RAG architecture?"
result=app.invoke({
    "messages": [HumanMessage(content=query)],
    "documents":None,
    "on_topic":"Unknown"
})


In [11]:
result

{'messages': [HumanMessage(content='What are the fundamental differences between utilizing a foundation model, LLM fine-tuning, and a RAG architecture?', additional_kwargs={}, response_metadata={}, id='dc44863f-1a99-45ef-8356-d22711933414'),
  AIMessage(content='Based on the provided documents, here are the fundamental differences between a foundation model and a RAG architecture:\n\n*   **Foundation Model:**\n    *   All training data is integrated directly into the parametric model of the LLM.\n    *   These models are typically trained on massive text corpora from the World Wide Web, such as the CommonCrawl dataset and the Pile.\n    *   Their creation and training demand immense resources and infrastructure, making it feasible only for very large organizations.\n    *   They are well-suited for a broad range of applications that do not rely on organizational or private data.\n\n*   **Retrieval-Augmented Generation (RAG) Architecture:**\n    *   RAG systems use the input sequence to